#  Notebook 05 — All Visualizations

Runs every plot in the project and saves to `outputs/plots/`.

Use this notebook to regenerate all figures.

In [ ]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / 'src').exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT / 'src'))

import warnings
from config_utils import load_config

warnings.filterwarnings('ignore')
cfg = load_config(PROJECT_ROOT / 'configs' / 'config.yaml')
print(f'Ready from {PROJECT_ROOT}')

## EDA Plots

In [ ]:
from visualize import run_all_eda_plots
run_all_eda_plots(cfg['_meta']['config_path'])
print(f"EDA plots done: {cfg['paths']['plots']}")

## Model Evaluation Plots

In [ ]:
import pandas as pd, numpy as np, joblib
from features import build_features_for_splits, get_feature_columns, apply_scaler, prepare_Xy
from visualize import (
    plot_confusion_matrix, plot_roc_curve, plot_precision_recall,
    plot_feature_importance, plot_actual_vs_predicted
)
train = pd.read_csv(cfg['paths']['train'], parse_dates=['date'])
val   = pd.read_csv(cfg['paths']['val'], parse_dates=['date'])
test  = pd.read_csv(cfg['paths']['test'], parse_dates=['date'])
_, _, test = build_features_for_splits(train=train, val=val, test=test)
feat_cols = get_feature_columns(cfg)
feat_cols = [c for c in feat_cols if c in test.columns]
# Classification
clf = joblib.load(cfg['paths']['model_clf'])
sc  = joblib.load(cfg['paths']['scaler'])
X_test, y_test = prepare_Xy(test, feat_cols, 'rain_today')
X_test_sc = apply_scaler(X_test, sc)
y_pred  = clf.predict(X_test_sc)
y_proba = clf.predict_proba(X_test_sc)[:,1]
plot_confusion_matrix(y_test, y_pred, cfg)
plot_roc_curve(y_test, y_proba, cfg)
plot_precision_recall(y_test, y_proba, cfg)
plot_feature_importance(feat_cols, clf.feature_importances_, 'XGBoost Classifier', cfg)
# Regression
reg = joblib.load(cfg['paths']['model_reg'])
sc2 = joblib.load(cfg['paths']['scaler'].replace('.pkl','_reg.pkl'))
X_r, y_r = prepare_Xy(test, feat_cols, 'PRECTOTCORR', log_transform=True)
X_r_sc = apply_scaler(X_r, sc2)
y_p = np.expm1(reg.predict(X_r_sc)).clip(min=0)
plot_actual_vs_predicted(np.expm1(y_r), y_p, 'XGBoost Regressor', cfg)
print(f"All plots saved to {cfg['paths']['plots']}")

## List All Saved Plots

In [ ]:
import os
plots = sorted(os.listdir(cfg['paths']['plots']))
for p in plots:
    print(p)